# TT-20 — SVR: Dự đoán cường độ chịu nén bê tông

**Bộ dữ liệu:** Concrete Compressive Strength (UCI, 1030 dòng × 8 đặc trưng)
**Mục tiêu:** dự đoán cường độ (MPa) ngay từ tỉ lệ phối trộn, thay vì chờ đủ 28 ngày nén mẫu.

Toàn bộ logic nằm trong `src/`; notebook này chạy lại từng bước và hiển thị kết quả.
Nếu chưa có dữ liệu, `load_raw()` sẽ tự tải từ UCI về thư mục `data/`.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import data as D
import train as T
from sklearn.model_selection import train_test_split

## 1. Nạp dữ liệu & khám phá

In [2]:
df = D.load_raw()
display(df.head())
display(df.describe().T[["mean", "std", "min", "max"]].round(2))

[data] 1030 dòng -> 1005 dòng sau khi bỏ trùng lặp, 9 cột


,Cement,BlastFurnaceSlag,FlyAsh,Water,Superplasticizer,CoarseAggregate,FineAggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28.0,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28.0,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270.0,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365.0,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360.0,44.296075


,mean,std,min,max
Cement,278.63,104.35,102.00,540.0
BlastFurnaceSlag,72.04,86.17,0.00,359.4
FlyAsh,55.54,64.21,0.00,200.1
Water,182.07,21.34,121.75,247.0
Superplasticizer,6.03,5.92,0.00,32.2
CoarseAggregate,974.38,77.58,801.00,1145.0
FineAggregate,772.69,80.34,594.00,992.6
Age,45.86,63.73,1.00,365.0
Strength,35.25,16.28,2.33,82.6


In [3]:
T.step1_eda(df)
for name in ["eda_scatter.png", "correlation_heatmap.png", "age_distribution.png"]:
    print(name)

[plot] reports/eda_scatter.png


[plot] reports/correlation_heatmap.png

=== Tương quan với cường độ (|r| giảm dần) ===
             Đặc trưng      r
    water_binder_ratio -0.611
          total_binder  0.598
               log_age  0.560
aggregate_binder_ratio -0.555
    water_cement_ratio -0.489
                Cement  0.488
      Superplasticizer  0.344
                   Age  0.337
                 Water -0.270
       sp_binder_ratio  0.222
         FineAggregate -0.186
       CoarseAggregate -0.145
      BlastFurnaceSlag  0.103
                FlyAsh -0.081


[plot] reports/age_distribution.png
eda_scatter.png
correlation_heatmap.png
age_distribution.png


## 2. Đặc trưng từ kiến thức miền

Model không tự nghĩ ra được tỉ lệ nước/xi măng — đây là kiến thức ngành xây dựng
(định luật Abrams: cường độ giảm theo hàm mũ khi w/c tăng). Tương tự, cường độ tăng
gần tuyến tính theo `log(tuổi)` chứ không theo `tuổi`.

In [4]:
d = D.add_domain_features(df)
display(d[D.DOMAIN_FEATURES].describe().T[["mean", "std", "min", "max"]].round(3))

X, y, cols = D.get_xy(df, use_domain=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=T.SEED)
print(f"Train {X_tr.shape} | Test {X_te.shape} | {len(cols)} đặc trưng")

,mean,std,min,max
water_cement_ratio,0.756,0.314,0.267,1.882
water_binder_ratio,0.473,0.126,0.235,0.900
total_binder,406.207,91.424,200.000,640.000
log_age,3.245,1.109,0.693,5.903
aggregate_binder_ratio,4.565,1.234,2.377,9.850
sp_binder_ratio,0.014,0.013,0.000,0.057


Train (804, 14) | Test (201, 14) | 14 đặc trưng


## 3. Baseline

In [5]:
T.step3_baseline(X_tr, X_te, y_tr, y_te)


=== BƯỚC 3 — Baseline ===
                      Model  RMSE test  R² test
DummyRegressor (trung bình)     17.297  -0.0028
          Linear Regression      7.384   0.8172


,Model,RMSE test,R² test
0,DummyRegressor (trung bình),17.297,-0.0028
1,Linear Regression,7.384,0.8172


## 4–5. SVR bắt buộc chuẩn hoá CẢ X LẪN y

`epsilon=0.1` mặc định được hiểu trên **thang đo của y**. Với y tính bằng MPa (2–83),
ε=0.1 gần như bằng 0 → mọi điểm đều nằm ngoài ống → mất sạch lợi ích của ε-insensitive loss
và số support vector tăng vọt. `TransformedTargetRegressor` xử lý việc scale y tự động.

In [6]:
T.step45_scaling(X_tr, X_te, y_tr, y_te)


=== BƯỚC 4+5 — Ảnh hưởng của chuẩn hoá (ε = 0.1 cố định) ===
              Cấu hình  RMSE test  R² test  Số SV  % SV
     Không scale gì cả      9.893   0.6720    790 98.3%
           Chỉ scale X      4.986   0.9167    770 95.8%
Scale cả X và y (đúng)      4.707   0.9257    474 59.0%
  Độ lệch chuẩn của y = 16.01 MPa -> ε=0.1 chỉ bằng 0.62% độ lệch chuẩn khi KHÔNG scale y.
  Sau khi scale y (std = 1), ε=0.1 tương đương ±1.60 MPa — hợp lý.
[plot] reports/scale_vs_noscale.png


,Cấu hình,RMSE test,R² test,Số SV,% SV
0,Không scale gì cả,9.893,0.6720,790,98.3%
1,Chỉ scale X,4.986,0.9167,770,95.8%
2,Scale cả X và y (đúng),4.707,0.9257,474,59.0%


## 6. So sánh kernel

In [7]:
T.step6_kernels(X_tr, X_te, y_tr, y_te)


=== BƯỚC 6 — So sánh kernel (C=100, ε=0.1) ===
      Kernel  RMSE test  R² test  Thời gian fit (s)
      linear      7.593   0.8068               3.86
poly (deg 2)      6.135   0.8739               2.37
poly (deg 3)      5.903   0.8832               8.69
         rbf      4.707   0.9257               0.26
[plot] reports/kernel_comparison.png


,Kernel,RMSE test,R² test,Thời gian fit (s)
0,linear,7.593,0.8068,3.86
1,poly (deg 2),6.135,0.8739,2.37
2,poly (deg 3),5.903,0.8832,8.69
3,rbf,4.707,0.9257,0.26


## 7–8. GridSearchCV + heatmap C × gamma

In [8]:
gs = T.step78_grid(X_tr, X_te, y_tr, y_te)
best = dict(kernel="rbf", **{k.split("__")[-1]: v for k, v in gs.best_params_.items()})
best


=== BƯỚC 7 — GridSearchCV (48 tổ hợp, 49.5s) ===
  Tham số tốt nhất : {'C': 10, 'epsilon': 0.1, 'gamma': 0.1}
  RMSE CV          : 4.910 MPa
[plot] reports/C_gamma_heatmap.png

=== BƯỚC 8 — Bảng RMSE CV theo (C, gamma) ===
     C  scale  0.01   0.1     1
   1.0  5.293 6.217 5.221 9.015
  10.0  4.938 5.707 4.910 8.523
 100.0  5.536 5.227 6.007 8.700
1000.0  6.107 5.249 6.195 8.987


{'kernel': 'rbf', 'C': 10, 'epsilon': 0.1, 'gamma': 0.1}

## 9. Model cuối: support vectors & phần dư

In [9]:
model = T.make_svr(**best).fit(X_tr, y_tr)
T.step9_support_vectors(model, X_tr, y_tr, X_te, y_te)


=== BƯỚC 9 — Support vectors & chất lượng model cuối ===
  Số support vector : 480/804  (59.7% dữ liệu train)
  SV chạm biên (|α| = C) : 178 (22.1%)
  ε = 0.1 (trên thang y đã chuẩn hoá) ≈ ±1.60 MPa
  RMSE train/test : 2.745 / 4.784 MPa
  R²   train/test : 0.9706 / 0.9233


[plot] reports/residual_analysis.png


{'n_support_vectors': 480,
 'pct_support_vectors': 59.701492537313435,
 'n_at_bound': 178,
 'rmse_test': 4.783948088963146,
 'r2_test': 0.9232852862787128}

## 10. Đặc trưng miền đóng góp bao nhiêu?

In [10]:
T.step10_domain_ablation(df, best)


=== BƯỚC 10 — Đóng góp của đặc trưng từ kiến thức miền ===
               Bộ đặc trưng  Số cột  RMSE test  R² test
            8 đặc trưng gốc       8      5.983   0.8800
Đầy đủ − water_cement_ratio      13      4.772   0.9237
  + đặc trưng miền (đầy đủ)      14      4.784   0.9233
  Cải thiện RMSE nhờ nhóm đặc trưng miền: +1.199 MPa (20.0%)
[plot] reports/domain_feature_gain.png


,Bộ đặc trưng,Số cột,RMSE test,R² test
0,8 đặc trưng gốc,8,5.983,0.8800
1,Đầy đủ − water_cement_ratio,13,4.772,0.9237
2,+ đặc trưng miền (đầy đủ),14,4.784,0.9233


## 11. SVR vs XGBoost vs Random Forest vs LinearSVR

In [11]:
T.step11_model_comparison(X_tr, X_te, y_tr, y_te, best)


=== BƯỚC 11 — So sánh mô hình ===
                   Model  RMSE test  MAE test  R² test  Train (s)  Predict (ms)
                 XGBoost      3.941     2.496   0.9479       0.47           2.0
SVR (rbf, đã tinh chỉnh)      4.784     2.989   0.9233       0.06           4.0
           Random Forest      4.875     3.455   0.9203       1.75          37.0
               LinearSVR      7.623     5.695   0.8052       0.02           0.3


[plot] reports/model_comparison.png


,Model,RMSE test,MAE test,R² test,Train (s),Predict (ms)
0,XGBoost,3.941,2.496,0.9479,0.47,2.0
1,"SVR (rbf, đã tinh chỉnh)",4.784,2.989,0.9233,0.06,4.0
2,Random Forest,4.875,3.455,0.9203,1.75,37.0
3,LinearSVR,7.623,5.695,0.8052,0.02,0.3


## 12. Hạn chế: SVR không mở rộng được

Nhân dữ liệu lên 1×→10× và đo thời gian train. Độ dốc log–log cho thấy chi phí
tăng xấp xỉ O(n²) — lý do SVR không dùng được cho dữ liệu lớn.

In [12]:
T.step12_scalability(X_tr, y_tr, best)

   1× | n=   804 | SVR    0.06s | RF   0.90s


   2× | n=  1608 | SVR    0.16s | RF   2.19s


   3× | n=  2412 | SVR    0.35s | RF   3.37s


   5× | n=  4020 | SVR    0.78s | RF   5.81s


   7× | n=  5628 | SVR    1.57s | RF   8.40s


  10× | n=  8040 | SVR    3.22s | RF  12.13s

  Độ dốc log-log của SVR ≈ 1.73 -> thời gian train ~ O(n^1.7)
  => Với 50.000+ dòng, SVR trở nên bất khả thi. Đây là hạn chế cốt lõi.


[plot] reports/thoi_gian_train.png

=== BƯỚC 12 — Khả năng mở rộng ===
Hệ số nhân  n mẫu  SVR (s)  RandomForest (s)
        1×    804     0.06              0.90
        2×   1608     0.16              2.19
        3×   2412     0.35              3.37
        5×   4020     0.78              5.81
        7×   5628     1.57              8.40
       10×   8040     3.22             12.13


,Hệ số nhân,n mẫu,SVR (s),RandomForest (s)
0,1×,804,0.06,0.90
1,2×,1608,0.16,2.19
2,3×,2412,0.35,3.37
3,5×,4020,0.78,5.81
4,7×,5628,1.57,8.40
5,10×,8040,3.22,12.13


## 13. Mở rộng — an toàn kết cấu

Dự báo cường độ **cao hơn** thực tế nguy hiểm hơn nhiều so với dự báo thấp hơn.
Mô hình hồi quy thường tối ưu trung bình có điều kiện → sai lệch hai phía xấp xỉ 50/50.
Dự báo phân vị 10% cho cận dưới an toàn để ra quyết định đổ móng.

In [13]:
T.step13_safety(model, X_tr, X_te, y_tr, y_te)


=== MỞ RỘNG — Phân tích an toàn kết cấu ===
                           Mô hình   RMSE Dự báo CAO hơn thực tế  Vượt > 5 MPa (nguy hiểm) Vượt > 5 MPa (%)
     SVR (trung bình có điều kiện)  4.784                  57.2%                        15             7.5%
GBR phân vị 10% (cận dưới an toàn) 11.541                  11.4%                         3             1.5%
  Dự báo phân vị thấp đánh đổi RMSE để gần như loại bỏ rủi ro đánh giá cao quá.
[plot] reports/safety_quantile.png


,Mô hình,RMSE,Dự báo CAO hơn thực tế,Vượt > 5 MPa (nguy hiểm),Vượt > 5 MPa (%)
0,SVR (trung bình có điều kiện),4.784,57.2%,15,7.5%
1,GBR phân vị 10% (cận dưới an toàn),11.541,11.4%,3,1.5%


## 14. Kết luận

- SVR chỉ hoạt động khi chuẩn hoá **cả X lẫn y**; bỏ qua scale y làm ε mất ý nghĩa.
- Đặc trưng từ kiến thức miền (w/c, log tuổi, tổng chất kết dính) là nguồn cải thiện lớn nhất —
  lớn hơn cả việc tinh chỉnh siêu tham số.
- Kernel RBF thắng linear rõ rệt → quan hệ phối trộn ↔ cường độ là phi tuyến.
- XGBoost thường nhỉnh hơn một chút và mở rộng tốt hơn nhiều; SVR chỉ phù hợp với
  dữ liệu vài nghìn dòng.
- Hạn chế: SVR không giải thích được từng dự đoán (cần SHAP/PDP riêng) và không mở rộng được.